In [ ]:
import numpy as np
import pandas as pd
import re
import warnings
from rbo import RankingSimilarity
warnings.filterwarnings('ignore')

In [ ]:
data_raw = pd.read_json('data/mrbilit_search.json')
data = data_raw[data_raw['ServiceType'].isin(['bus', 'taxi'])].copy().reset_index(drop=True)
data.head()

,ServiceType,TypedStrings,AcceptString
0,bus,[],تهران
1,bus,[تهران],تهران
2,bus,"[, ]",اصفهان
3,taxi,[اصفهان],اصفهان
4,bus,[تهران],تهران


In [ ]:
test = pd.read_json('data/test_data.json')
test.head()

,Typed
0,تهد
1,نها
2,ساوه
3,یا
4,بندر انزلی


In [ ]:
cities = pd.read_csv('data/iran_cities.csv')
cities.head(3)

,City EN,City FA,Province,Countise,District,Latitude,Longitude,Area,Elevation,2016 Census,2011 Census,Wikipedia EN,Wikipedia FA,GeoHack
0,Ab Bar,آب‌بر,زنجان,طارم,مرکزی,36.918100,48.96470,709.0,NaN,8091.0,6725.0,https://en.wikipedia.org/wiki/Ab_Bar,https://fa.wikipedia.org/wiki/%D8%A2%D8%A8%E2%...,https://geohack.toolforge.org/geohack.php?lang...
1,Ab Pakhsh,آب‌پخش,بوشهر,دشتستان,آب‌پخش,29.363706,51.07018,251204.0,7003.0,18913.0,17238.0,https://en.wikipedia.org/wiki/Ab_Pakhsh,https://fa.wikipedia.org/wiki/%D8%A2%D8%A8%E2%...,https://geohack.toolforge.org/geohack.php?lang...
2,Abad,آباد,بوشهر,تنگستان,مرکزی,29.028300,51.24910,45.0,NaN,3787.0,3787.0,"https://en.wikipedia.org/wiki/Abad,_Bushehr",https://fa.wikipedia.org/wiki/%D8%A2%D8%A8%D8%...,https://geohack.toolforge.org/geohack.php?lang...


In [ ]:
typo_df = pd.read_csv('data/typo_char.csv')
typo_df.head(10)

,EN,FA
0,a,ش
1,b,ذ
2,c,ز
3,d,ی
4,e,ث
5,f,ب
6,g,ل
7,h,ا
8,i,ه
9,j,ت


In [ ]:
data['CityBase'] = (
    data['AcceptString']
    .astype(str)
    .str.split(' - ')
    .str[0]
    .str.strip()
)

sample = data[data['AcceptString'] != data['CityBase']].head(5)

accept_freq = data['AcceptString'].value_counts()

city_freq = data['CityBase'].value_counts()

print('Top 20 most selected cities (base name):')
print(city_freq.head(20).to_string())

print('Typo mapping columns:', typo_df.columns.tolist())
print(typo_df.head(15))

col_en = typo_df.columns[0] 
col_fa = typo_df.columns[1]

typo_map = dict(zip(
    typo_df[col_en].astype(str).str.lower(),
    typo_df[col_fa].astype(str)
))
print('\nTypo mapping dict (sample):', dict(list(typo_map.items())[:10]))

all_accept_strings = accept_freq.index.tolist()

all_cities = city_freq.index.tolist()

fa_to_en = dict(zip(
    cities['City FA'].astype(str).str.strip(),
    cities['City EN'].astype(str).str.strip().str.lower()
))

en_to_fa = dict(zip(
    cities['City EN'].astype(str).str.strip().str.lower(),
    cities['City FA'].astype(str).str.strip()
))

print(f'Total unique AcceptStrings: {len(all_accept_strings)}')
print(f'Total unique base cities: {len(all_cities)}')
print(f'English↔Persian city mappings: {len(fa_to_en)}')

def convert_keyboard_typo(text):

    result = ''
    for ch in text.lower():
        result += typo_map.get(ch, ch)
    return result


def normalize(text):

    text = str(text).strip().lower()
    text = text.replace('ي', 'ی').replace('ك', 'ک').replace('ة', 'ت')
    text = text.replace('\u200c', '') 
    return text


def is_mostly_english(text):

    ascii_chars = sum(1 for c in text if c.isascii() and c.isalpha())
    return ascii_chars > len(text) * 0.5

typed_to_accept = {}  

for _, row in data.iterrows():
    accept = str(row['AcceptString']).strip()
    typed_list = row['TypedStrings']
    if not isinstance(typed_list, list):
        continue
    for typed in typed_list:
        typed_norm = normalize(str(typed).strip())
        if len(typed_norm) == 0:
            continue
        if typed_norm not in typed_to_accept:
            typed_to_accept[typed_norm] = {}
        typed_to_accept[typed_norm][accept] = typed_to_accept[typed_norm].get(accept, 0) + 1

Top 20 most selected cities (base name):
CityBase
تهران       6252
اصفهان      2446
مشهد        2190
شیراز       1587
قم           978
یزد          977
اهواز        881
بندرعباس     712
رشت          710
تبریز        621
کرج          620
کرمان        607
ساری         492
گرگان        421
اراک         411
کرمانشاه     370
همدان        355
زنجان        321
بابل         280
خرم آباد     261
Typo mapping columns: ['EN', 'FA']
   EN FA
0   a  ش
1   b  ذ
2   c  ز
3   d  ی
4   e  ث
5   f  ب
6   g  ل
7   h  ا
8   i  ه
9   j  ت
10  k  ن
11  l  م
12  m  ئ
13  n  د
14  o  خ

Typo mapping dict (sample): {'a': 'ش', 'b': 'ذ', 'c': 'ژ', 'd': 'ی', 'e': 'ث', 'f': 'ب', 'g': 'ل', 'h': 'آ', 'i': 'ه', 'j': 'ت'}
Total unique AcceptStrings: 352
Total unique base cities: 328
English↔Persian city mappings: 1223


In [ ]:
def suggest(typed_raw, n=5):
    typed_raw = str(typed_raw).strip()
    typed_norm = normalize(typed_raw)
    typed_lower = typed_raw.lower()

    typed_typo_corrected = normalize(convert_keyboard_typo(typed_raw))
    
    scores = {} 

    if typed_norm in typed_to_accept:
        for accept, count in typed_to_accept[typed_norm].items():
            scores[accept] = scores.get(accept, 0) + count * 50

    if typed_typo_corrected != typed_norm and typed_typo_corrected in typed_to_accept:
        for accept, count in typed_to_accept[typed_typo_corrected].items():
            scores[accept] = scores.get(accept, 0) + count * 40

    for accept_str in all_accept_strings:
        accept_norm = normalize(accept_str)
        base_city = accept_str.split(' - ')[0].strip()
        base_norm = normalize(base_city)
        freq = accept_freq.get(accept_str, 1)
        
        if typed_norm == accept_norm or typed_norm == base_norm:
            scores[accept_str] = scores.get(accept_str, 0) + 10000 + freq
            continue
    
        if len(typed_norm) > 0 and base_norm.startswith(typed_norm):
            scores[accept_str] = scores.get(accept_str, 0) + 1000 + freq
        

        elif len(typed_norm) > 1 and typed_norm in base_norm:
            scores[accept_str] = scores.get(accept_str, 0) + 400 + freq
        
        if (typed_typo_corrected != typed_norm 
                and len(typed_typo_corrected) > 0 
                and base_norm.startswith(typed_typo_corrected)):
            scores[accept_str] = scores.get(accept_str, 0) + 800 + freq

        if is_mostly_english(typed_raw) and len(typed_lower) >= 2:
            en_name = fa_to_en.get(base_city, '').lower()
            if en_name and en_name.startswith(typed_lower):
                scores[accept_str] = scores.get(accept_str, 0) + 600 + freq
            elif en_name and typed_lower in en_name:
                scores[accept_str] = scores.get(accept_str, 0) + 300 + freq
    
    sorted_accepts = sorted(scores.items(), key=lambda x: -x[1])
    
    suggestions = []
    seen_bases = set()
    
    for accept_str, score in sorted_accepts:
        base = accept_str.split(' - ')[0].strip()

        if base not in seen_bases:
            suggestions.append(accept_str)
            seen_bases.add(base)
        if len(suggestions) >= n:
            break

    if len(suggestions) < n:
        for accept_str in all_accept_strings:
            base = accept_str.split(' - ')[0].strip()
            if base not in seen_bases:
                suggestions.append(accept_str)
                seen_bases.add(base)
            if len(suggestions) >= n:
                break
    
    return suggestions[:n]

In [ ]:
act = ['A', 'B', 'C']
pred = ['B', 'A', 'D']
rbo = RankingSimilarity(act, pred).rbo()
print(rbo)

0.5555555555555555


In [11]:
print('Verification against notebook examples:')
print(f'  "تهد" → {suggest("تهد")}')
print(f'  "قا"  → {suggest("قا")}')
print(f'  "بوی" → {suggest("بوی")}')
print(f'  "dcn" → {suggest("dcn")}')
print(f'  "ت"   → {suggest("ت")}')

Verification against notebook examples:
  "تهد" → ['تهران', 'کلاله', 'اصفهان', 'مشهد', 'شیراز']
  "قا"  → ['قائم شهر', 'قائن', 'قایمیه', 'قاینات', 'ساری']
  "بوی" → ['بویراحمد', 'بویین ومیاندشت', 'تهران', 'اصفهان', 'مشهد']
  "dcn" → ['یزد', 'تهران', 'اصفهان', 'مشهد', 'شیراز']
  "ت"   → ['تهران', 'تبریز', 'تنکابن', 'تربت حیدریه', 'تفرش']


In [12]:
print(f'Generating suggestions for {len(test)} test samples...')

rows = []
for i, typed in enumerate(test['Typed']):
    suggestions = suggest(str(typed), n=5)
    # Pad with fallback cities if needed
    while len(suggestions) < 5:
        for city in all_accept_strings:
            if city not in suggestions:
                suggestions.append(city)
                break
    rows.append(suggestions[:5])
    if (i + 1) % 100 == 0:
        print(f'  Processed {i+1}/{len(test)}')

print('Done!')

Generating suggestions for 519 test samples...
  Processed 100/519
  Processed 200/519
  Processed 300/519
  Processed 400/519
  Processed 500/519
Done!


In [13]:
submission = pd.DataFrame(
    rows,
    columns=['Suggestion0', 'Suggestion1', 'Suggestion2', 'Suggestion3', 'Suggestion4']
)

submission.head()

,Suggestion0,Suggestion1,Suggestion2,Suggestion3,Suggestion4
0,تهران,کلاله,اصفهان,مشهد,شیراز
1,نهاوند,تهران,اصفهان,مشهد,شیراز
2,ساوه,تفرش,تهران,قم,اصفهان
3,یاسوج,میاندوآب,میانه,شهریار,رویان
4,بندر انزلی,عسلویه,بندر ترکمن,تهران,اصفهان
